In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
import datetime
import os

os.makedirs("data", exist_ok=True)
print("All imports done!")

All imports done!


In [13]:
my_categories = {
    "mobiles":    "https://www.flipkart.com/search?q=smartphones&page={}",
    "laptops":    "https://www.flipkart.com/search?q=laptops&page={}",
    "headphones": "https://www.flipkart.com/search?q=headphones&page={}",
}

pages_to_scrape = 5
wait_time = 4

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

print("Settings ready!")

Settings ready!


In [14]:
def clean_price(text):
    if text:
        return text.replace("₹", "").replace(",", "").strip()
    return None

def scrape_page(driver, url, category):
    products = []

    driver.get(url)
    time.sleep(4)

    soup = BeautifulSoup(driver.page_source, "lxml")

    cards = soup.find_all("div", class_="jIjQ8S")
    if not cards:
        cards = soup.find_all("div", class_="nZIRY7")
    print(f"cards found: {len(cards)}", end=" | ")
    for card in cards:
        try:
            
            name_tag = (
                card.find("div", class_="RG5Slk") or
                card.find("div", class_="RGLWAk")
            ) 

            product_name = name_tag.get_text().strip() if name_tag else None

            price_container = card.find("div", class_="QiMO5r")
            selling_price = None
            mrp = None
            if price_container:

                all_prices = price_container.find_all(
                    string=lambda t: "₹" in str(t) and any(c.isdigit() for c in str(t))
                )
                if len(all_prices) >= 1:
                    selling_price = clean_price(all_prices[0])
                if len(all_prices) >= 2:
                    mrp = clean_price(all_prices[1])

            discount_tag = card.find("div", class_="HQe8jr")
            discount_shown = None
            if discount_tag:
                discount_shown = (
                    discount_tag.get_text()
                    .replace("% off", "")
                    .replace("%", "")
                    .strip()
                )

            rating_tag = card.find("span", class_="CjyrHS")
            rating = rating_tag.get_text().strip() if rating_tag else None

            specs_tag = card.find("div", class_="CMXw7N")
            specs = specs_tag.get_text().strip() if specs_tag else None

            if not product_name or not selling_price:
                continue

            real_discount = None
            if selling_price and mrp:
                try:
                    sp = float(selling_price)
                    mp = float(mrp)
                    if mp > 0 and mp >= sp:
                        real_discount = round(((mp - sp) / mp) * 100, 2)
                except:
                    pass

            products.append({
                "product_name":        product_name,
                "selling_price_inr":   selling_price,
                "mrp_inr":             mrp,
                "discount_as_listed":  discount_shown,
                "discount_calculated": real_discount,
                "star_rating":         rating,
                "specs":               specs,
                "category":            category,
                "platform":            "Flipkart",
                "source_url":          url,
                "scraped_at":          datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })

        except Exception:
            continue

    return products

print("Scraping function ready!")

Scraping function ready!


In [15]:
all_products = []

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

print("Chrome opened — starting scrape...\n")

for category, url_template in my_categories.items():
    print(f"\nCategory: {category}")

    for page_num in range(1, pages_to_scrape + 1):
        url = url_template.format(page_num)
        print(f"  Page {page_num}/{pages_to_scrape} ... ", end="")

        page_data = scrape_page(driver, url, category)
        all_products.extend(page_data)

        print(f"total so far: {len(all_products)}")
        time.sleep(wait_time)

driver.quit()
print(f"\nChrome closed.")
print(f"Total products collected: {len(all_products)}")

Chrome opened — starting scrape...


Category: mobiles
  Page 1/5 ... cards found: 24 | total so far: 24
  Page 2/5 ... cards found: 0 | total so far: 24
  Page 3/5 ... cards found: 24 | total so far: 48
  Page 4/5 ... cards found: 24 | total so far: 72
  Page 5/5 ... cards found: 24 | total so far: 96

Category: laptops
  Page 1/5 ... cards found: 24 | total so far: 120
  Page 2/5 ... cards found: 24 | total so far: 144
  Page 3/5 ... cards found: 24 | total so far: 168
  Page 4/5 ... cards found: 24 | total so far: 192
  Page 5/5 ... cards found: 24 | total so far: 216

Category: headphones
  Page 1/5 ... cards found: 10 | total so far: 226
  Page 2/5 ... cards found: 10 | total so far: 236
  Page 3/5 ... cards found: 10 | total so far: 246
  Page 4/5 ... cards found: 10 | total so far: 256
  Page 5/5 ... cards found: 10 | total so far: 266

Chrome closed.
Total products collected: 266


In [16]:
df = pd.DataFrame(all_products)

before = len(df)
df.drop_duplicates(
    subset=["product_name", "selling_price_inr", "category"],
    inplace=True
)
after = len(df)

today = datetime.datetime.now().strftime("%d%b%Y")
filename = f"data/flipkart_raw_{today}.csv"
df.to_csv(filename, index=False, encoding="utf-8-sig")

print(f"Rows collected      : {before}")
print(f"After deduplication : {after}")
print(f"Saved to            : {filename}")
print()
df.head(10)

Rows collected      : 266
After deduplication : 212
Saved to            : data/flipkart_raw_22Apr2026.csv



,product_name,selling_price_inr,mrp_inr,discount_as_listed,discount_calculated,star_rating,specs,category,platform,source_url,scraped_at
0,"MOTOROLA g06 power (Pantone tapestry, 64 GB)",9999,None,None,NaN,4.3,4 GB RAM | 64 GB ROM | Expandable Upto 1 TB17....,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-22 16:11:38
1,"realme P4 Lite 5G (Mosaic Blue, 64 GB)",12749,19999,36,36.25,4.4,4 GB RAM | 64 GB ROM17.27 cm (6.8 inches) HD+ ...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-22 16:11:38
2,"Ai+ Pulse 1 (Sparkle Red, 64 GB)",7999,None,None,NaN,4.3,4 GB RAM | 64 GB ROM | Expandable Upto 1 TB17....,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-22 16:11:38
3,"Ai+ Pulse 1 (Blue, 64 GB)",7999,None,None,NaN,4.3,4 GB RAM | 64 GB ROM | Expandable Upto 1 TB17....,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-22 16:11:38
4,"realme P4 Lite 5G (Mosaic Green, 128 GB)",15749,22999,31,31.52,4.5,6 GB RAM | 128 GB ROM17.27 cm (6.8 inches) HD+...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-22 16:11:38
5,"POCO C85x (Elite Black, 64 GB)",11499,13499,14,14.82,4.2,4 GB RAM | 64 GB ROM | Expandable Upto 2 TB17....,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-22 16:11:38
6,"MOTOROLA g35 5G (Leaf Green, 128 GB)",12499,None,None,NaN,4.2,4 GB RAM | 128 GB ROM | Expandable Upto 1 TB17...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-22 16:11:38
7,"MOTOROLA g57 power 5G (Pantone Regatta, 128 GB)",15999,17999,11,11.11,4.4,8 GB RAM | 128 GB ROM17.07 cm (6.72 inch) Full...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-22 16:11:38
8,"realme P4x 5G (Matte Silver, 128 GB)",16499,17999,8,8.33,4.4,6 GB RAM | 128 GB ROM17.07 cm (6.72 inch) Full...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-22 16:11:38
9,"Ai+ Pulse 2 (Blue, 64 GB)",8499,10999,22,22.73,4.6,4 GB RAM | 64 GB ROM | Expandable Upto 1 TB17....,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-22 16:11:38


In [17]:
print("=== DATASET SUMMARY ===\n")
print(f"Total rows: {len(df)}")
print(f"\nRows per category:")
print(df["category"].value_counts())
print(f"\nRows per platform:")
print(df["platform"].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nDiscount range:")
print(df["discount_calculated"].describe())

=== DATASET SUMMARY ===

Total rows: 212

Rows per category:
mobiles       96
laptops       82
headphones    34
Name: category, dtype: int64

Rows per platform:
Flipkart    212
Name: platform, dtype: int64

Missing values:
product_name            0
selling_price_inr       0
mrp_inr                25
discount_as_listed     25
discount_calculated    25
star_rating             1
specs                  34
category                0
platform                0
source_url              0
scraped_at              0
dtype: int64

Discount range:
count    187.000000
mean      25.894920
std       22.841626
min        1.610000
25%       10.200000
50%       18.860000
75%       33.900000
max       90.190000
Name: discount_calculated, dtype: float64
